# **Interactive Building Extraction with SAM3**

This notebook demonstrates how to segment remote sensing images interactively using the Segment Anything Model 3 (SAM3). For detiled model impementation and further resources please consult Meta's page [HERE](https://ai.meta.com/research/sam3/) and the SAM3 paper [HERE](https://ai.meta.com/research/sam3/)

# **Installation**

This notebook accesses a SAM 3 impementation wraped to segmentgeopatial implemetation. For further details, integration with QGIS and ArcGIS pro environemnt, please consult from the [SOURCE](https://github.com/opengeos/segment-geospatial)

In [ ]:
%pip install "segment-geospatial[samgeo3]"
%pip install transformers==5.0.0rc0
%pip install -U huggingface_hub
%pip install kagglehub

## **Import necessary libraries**


In [ ]:
import leafmap  # for interactive visualization
from samgeo import SamGeo3  # A wrapper library for accessing SAM3 for inetractive segmentation
from sys import path
from glob import glob
import os
import rasterio      # for remotely image handeling
from pathlib import Path
from rasterio.merge import merge
import kagglehub                 # To access the data stored in the Kaggle platform
import matplotlib.pyplot as plt

## **Check if cuda GPU available**

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

## **Connect to Google Drive for user data access**

This will help to access and load custome files and processed outputs including trained model weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root_folder = "/content/drive/MyDrive/Deep-Learning-Building-Extraction-master"

Mounted at /content/drive


## **Access Very High Resolution Data**

For this exercise, the dataset is saved in [Kaggle](https://www.kaggle.com/) plateform. The dataset is observed at Kakuma IDP camp and Kutupalong Refugee camps and was accessed from [Open Aerial Maps](https://openaerialmap.org/).


In [ ]:
dataset_path = kagglehub.dataset_download("getachewworkineh/kakuma-ceos-training")
print("Path to dataset files:", dataset_path)
print(f"The folder contains the following files")
os.listdir(dataset_path)

## **Visualize the image for visual inspection**

In [ ]:
image = os.path.join(dataset_path, "kutupalong_a.tif")
os.path.exists(image)

True

In [ ]:
m = leafmap.Map()
m.add_raster(image, layer_name="Satellite image")
m

## **Request access to SAM3**

To use SAM3, we need to request access from Hugging Face: https://huggingface.co/facebook/sam3

Once we have access, we have to get the access token and past on the requested hugging face login and past the token.

In [ ]:
from huggingface_hub import login
login()

## **Initialize SAM3**

When initializing SAM3, you can choose the backend from "meta", or "transformers".

In [ ]:
sam3 = SamGeo3(
    backend="transformers",
    device=None,
    checkpoint_path=None,
    load_from_HF=True
)

## **Set the image**

You can set the image by either passing the image path or the image URL. At the same time we can optimize mask threshold and confidence threshold which are the most important hyperparameters in tunning proper object masks

In [ ]:
# sam3.set_confidence_threshold(0.5)
# sam3.mask_threshold = 0.25
# print(sam3.mask_threshold)
sam3.set_image(image)

## **Generate masks with text prompt**

In [ ]:
sam3.generate_masks(prompt="building")

Found 69 objects.


In [ ]:
sam3.show_masks(figsize=(10, 10), unique=True)

In [ ]:
plt.figure(figsize = (10,10))
plt.imshow(sam3.image)
plt.axis("off")
plt.show()

***Note:***
As we can clearly see the performance is not good. This is mainly affected by the tile size and the diversity of buildings itself in the scene. The solution, work in a more reduced tile size, that can profoundly reduce the bulding spectral diversity per scene.

## **Interactive segmentation**

Interactively segment using text promt or geometric promts (box promts). The underlying architecture is the same except chnaging the promt type.

In [ ]:
sam3.show_map(height="1000px", min_size=10)

## **Tile the image**

Tile the biger image into smaller chips, that can reduce building diversity, and increase the resolution. For this, we will use a custom function.

In [ ]:
import sys
sys.path.append(f"{root_folder}/scripts")
from tiling_and_visualization import tile_raster, tile_visualize
from prediction_to_geodata import binary_raster_to_vector

In [ ]:
tile_visualize(raster_file=image,
               tile_height=512,
               tile_width=512,
               stride_y=506,
               stride_x=506,
               with_tiles=True)

In [ ]:
tile_raster(input_raster=image,
            output_dir=f"{root_folder}/raw_dataset/sam_tile",
            tile_size=512,
            stride=506,
            input_mask_file=None,
            background_value=0)

Raster dimensions: 3217 x 4970
Number of tiles including background: 6 x 9

Successfully created 70 chips


## **Automate on small samples**

Provided that tile size is important parameter for speed and controling the diversity of buildings in the scene, the prediction will be run on smaller tiles with less divers buildings

In [ ]:
# sam3.set_confidence_threshold(0.5)
# sam3.mask_threshold = 0.5

root = f"{root_folder}/raw_dataset/sam_tile/images"
save_dir  = f"{root_folder}/raw_dataset/sam_tile/labels"
files = sorted(glob(root + "/*.tif"))
if not os.path.exists(save_dir):
  os.makedirs(save_dir)
for file in files:
  sam3.set_image(file)
  sam3.generate_masks(prompt="building")
  try:
    sam3.save_masks(f"{save_dir}/masks_{os.path.basename(file)}")
  except:
    pass

In [ ]:
print(list(glob(f"{save_dir}/*.tif")))

In [ ]:
raster_files = list(glob(f"{save_dir}/*.tif"))
paths = [rasterio.open(fls) for fls in raster_files]
dest, output_transform = merge(paths, indexes=1, method="first")
dest = dest>=1
dest = dest.astype(int)

with rasterio.open(raster_files[0]) as src:
    out_meta = src.meta.copy()
    out_meta.update(
        {
            "driver": "GTiff",
            "height": dest.shape[1],
            "width": dest.shape[2],
            "transform": output_transform,
            "count": 1
        }
    )
save_file = f"{save_dir}/merged_final.tif"
with rasterio.open(save_file, "w", **out_meta) as fp:
    fp.write(dest)

In [ ]:
plt.imshow(dest[0])
plt.show()

In [ ]:
save_file = f"{save_dir}/merged_final.tif"
m = leafmap.Map()
m.add_raster(save_file, layer_name="segmented")
m

In [ ]:
gdf = binary_raster_to_vector(
        save_file,
        output_path=f"{save_dir}/merged_final.geojson",
        mask_value=1,
        simplify_tolerance=None,
        min_area=2.0
    )

print(f"Found {len(gdf)} polygons")

    # Plot the result
if len(gdf) > 0:
    gdf.plot(figsize=(10, 10))
    import matplotlib.pyplot as plt
    plt.show()

In [ ]:
save_file = f"{save_dir}/merged_final.geojson"
m = leafmap.Map()
m.add_vector(save_file, layer_name="vector_file")
m

In [ ]:
save_file = f"{save_dir}/merged_final.tif"
gdf_filtered_area = binary_raster_to_vector(
                            save_file,
                            output_path=f"{save_dir}/merged_final_gen.geojson",
                            remove_holes=True,
                            min_hole_area=2,  # Remove holes smaller than 50 square units
                            simplify_tolerance=0.9,
                            min_area=1
                      )

Saved 1276 polygons to /content/drive/MyDrive/Deep-Learning-Building-Extraction-master/raw_dataset/sam_tile/labels/merged_final_gen.shp


In [ ]:
save_file = f"{save_dir}/merged_final.geojson"
save_file1 = f"{save_dir}/merged_final_gen.geojson"
m = leafmap.Map()
m.add_vector(save_file, layer_name="vector_file")
m.add_vector(save_file1, layer_name="1vector_file")
m